# Notebook 14C — Fannie Mae target validation
## Stage 3A–3C: Target contract and eligibility diagnostics
Use the same sample as Notebook 14B. Compare candidate event and eligibility rules without assigning final labels. Do not change Freddie artifacts or fit models.

## 1. Verify the Freddie target contract
Check the existing source code rather than redefining the original target. The first scheduled payment month is month 0; month 23 is the last included month. Terminal events are timed by reporting month.

In [1]:
from pathlib import Path
import ast, hashlib, json, re, zipfile
from collections import Counter
from functools import lru_cache
from datetime import datetime, timezone
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
SOURCE = ROOT / 'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
FREDDIE = ROOT / 'src/freddie_performance.py'
assert SOURCE.is_file() and FREDDIE.is_file()
source_text = FREDDIE.read_text(encoding='utf-8')
tree = ast.parse(source_text)
constants = {t.id: ast.literal_eval(node.value) for node in tree.body if isinstance(node, ast.Assign) for t in node.targets if isinstance(t, ast.Name) and t.id == 'CREDIT_EVENT_ZERO_BALANCE_CODES'}
assert constants['CREDIT_EVENT_ZERO_BALANCE_CODES'] == {'02','03','09','15'}
functions = {node.name: ast.get_source_segment(source_text,node) for node in tree.body if isinstance(node,ast.FunctionDef)}
for name in ['prepare_performance_timeline','create_horizon_targets']:
    print(functions[name])
freddie_sha = hashlib.sha256(FREDDIE.read_bytes()).hexdigest()
print('Freddie source SHA256:', freddie_sha)

def prepare_performance_timeline(
    performance: pd.DataFrame,
    origination: pd.DataFrame,
) -> pd.DataFrame:
    """Add a first-payment-relative month index and event flags."""

    validate_performance_for_target(
        performance=performance,
        origination=origination,
    )

    first_payment = origination[
        ["LOAN IDENTIFIER", "FIRST PAYMENT DATE"]
    ].copy()
    first_payment["LOAN IDENTIFIER"] = first_payment[
        "LOAN IDENTIFIER"
    ].astype("string")

    if first_payment["LOAN IDENTIFIER"].duplicated().any():
        raise ValueError(
            "Origination data contains duplicate loan identifiers."
        )

    if (~_valid_yyyymm(first_payment["FIRST PAYMENT DATE"])).any():
        raise ValueError(
            "Origination data contains invalid FIRST PAYMENT DATE values."
        )

    timeline = performance.copy()
    timeline["LOAN IDENTIFIER"] = timeline[
        "LOAN IDENTIFIER"
    ].astype("string")
    timeline = timeline.merge(
   

## 2. Record the candidate Fannie event definitions
Reference: [Fannie Mae glossary, July 29, 2024](https://capitalmarkets.fanniemae.com/sites/capmrkt/files/2024-07/crt-file-layout-and-glossary.pdf). The exact local release is unverified. The mapping below is a working hypothesis for diagnostics, not final approval. Freddie RA is counted as serious delinquency; Fannie XX is unknown and is not its equivalent. Repurchase ends observation but is not proof of a non-event. Review code definitions against the local release documentation before target construction.

In [2]:
ADVERSE = {'02','03','09','15'}
NON_ADVERSE = {'01','16','96'}
REPURCHASE = {'06'}
code_map = pd.DataFrame([
    ('01','Prepayment or maturity','non-adverse candidate'),
    ('02','Third-party sale','adverse candidate'),
    ('03','Short sale','adverse candidate'),
    ('06','Repurchase','separate eligibility review'),
    ('09','Deed-in-lieu or REO disposition','adverse candidate'),
    ('15','Non-performing note sale','adverse candidate'),
    ('16','Reperforming note sale','non-adverse candidate'),
    ('96','Non-credit removal','non-adverse candidate')], columns=['code','working_definition','candidate_treatment'])
print(code_map.to_string(index=False))

code              working_definition         candidate_treatment
  01          Prepayment or maturity       non-adverse candidate
  02                Third-party sale           adverse candidate
  03                      Short sale           adverse candidate
  06                      Repurchase separate eligibility review
  09 Deed-in-lieu or REO disposition           adverse candidate
  15        Non-performing note sale           adverse candidate
  16          Reperforming note sale       non-adverse candidate
  96              Non-credit removal       non-adverse candidate


## 3. Recreate the reviewed sample
Scan the file using the same hash, physical positions and memory guard as Notebook 14B. Stop if counts differ from the reviewed run.

In [3]:
SIZE = SOURCE.stat().st_size
WIDTH = 113
assert SIZE == 6928164686, 'Source differs from reviewed file.'
POSITIONS = {'loan_id':2,'report':3,'orig':14,'first_payment':15,'age':16,'delinq':40,'payment_history':41,'modification':42,'zero_code':44,'zero_month':45}
SAMPLE_MODULUS = 100
MAX_SAMPLE_ROWS = 1_000_000
SOURCE_SIGNATURE = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
@lru_cache(maxsize=100_000)
def selected(loan_id):
    return int.from_bytes(hashlib.blake2b(loan_id,digest_size=8,person=b'Fannie14v1').digest(),'big') % SAMPLE_MODULUS == 0
def scan_source(path, expected_width, progress_every=2_000_000):
    counts = Counter()
    sampled = []
    widths_seen = Counter()
    id_pattern = re.compile(rb'[A-Za-z0-9]{1,12}')
    indexes = [p-1 for p in POSITIONS.values()]
    with path.open('rb') as stream:
        for line_number, raw in enumerate(stream, 1):
            parts = raw.rstrip(b'\r\n').split(b'|')
            if line_number == 1:
                parts[0] = parts[0].removeprefix(b'\xef\xbb\xbf')
            counts['physical_rows'] += 1
            widths_seen[len(parts)] += 1
            if len(parts) != expected_width:
                counts['wrong_width'] += 1
                continue
            loan_id = parts[1].strip()
            if not id_pattern.fullmatch(loan_id):
                counts['invalid_loan_id'] += 1
                continue
            counts['valid_id_rows'] += 1
            if selected(loan_id):
                sampled.append(tuple(parts[i].strip().decode('ascii') for i in indexes))
                if len(sampled) > MAX_SAMPLE_ROWS:
                    raise RuntimeError('Sample exceeds memory guard; increase SAMPLE_MODULUS and rerun the entire scan.')
            if progress_every and line_number % progress_every == 0:
                print(f'{line_number:,} rows scanned; {len(sampled):,} sampled rows')
    return pd.DataFrame(sampled, columns=list(POSITIONS)), counts, widths_seen

history, scan_counts, full_widths = scan_source(SOURCE, WIDTH)
assert SOURCE_SIGNATURE == (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns), 'Source changed during scan.'
print(dict(scan_counts), dict(full_widths))
assert not scan_counts['wrong_width'] and not scan_counts['invalid_loan_id'], 'Malformed records found; review before continuing.'
assert not history.empty, 'No sampled rows'
print('Sample rows:', len(history), '| Sample distinct loans:', history.loan_id.nunique())
assert scan_counts['physical_rows'] == 23563399
assert len(history) == 245552 and history.loan_id.nunique() == 4749, 'Sample differs from reviewed Stage 2.' 

2,000,000 rows scanned; 22,660 sampled rows
4,000,000 rows scanned; 43,404 sampled rows
6,000,000 rows scanned; 63,865 sampled rows
8,000,000 rows scanned; 83,683 sampled rows
10,000,000 rows scanned; 103,966 sampled rows
12,000,000 rows scanned; 123,590 sampled rows
14,000,000 rows scanned; 145,624 sampled rows
16,000,000 rows scanned; 165,802 sampled rows
18,000,000 rows scanned; 186,285 sampled rows
20,000,000 rows scanned; 206,376 sampled rows
22,000,000 rows scanned; 229,575 sampled rows
{'physical_rows': 23563399, 'valid_id_rows': 23563399} {113: 23563399}
Sample rows: 245552 | Sample distinct loans: 4749


## 4. Check dates and observed codes
Keep unknown statuses separate. Check all dates and duplicate loan-month records before comparing candidate rules.

In [4]:
def month_index(series):
    valid = series.str.fullmatch(r'(0[1-9]|1[0-2])(19|20)\d{2}')
    safe = series.where(valid)
    return (pd.to_numeric(safe.str[2:]) * 12 + pd.to_numeric(safe.str[:2]) - 1).astype('Int64')

def month_text(i):
    if pd.isna(i): return 'missing'
    return f'{int(i)//12:04d}-{int(i)%12+1:02d}'

for name in ['report','orig','first_payment','zero_month']:
    history[name+'_idx'] = month_index(history[name])
    print(name, 'blank:', int(history[name].eq('').sum()), 'invalid nonblank:', int((history[name].ne('') & history[name+'_idx'].isna()).sum()))

history['numeric_delinq'] = pd.to_numeric(history.delinq.where(history.delinq.str.fullmatch(r'\d{1,2}')), errors='coerce')
valid_report = history[history.report_idx.notna()]
# Count duplicates without discarding conflicting rows.
duplicate_counts = valid_report.groupby(['loan_id','report_idx']).size()
duplicate_groups = duplicate_counts[duplicate_counts > 1]
print('Duplicate loan-month groups:', len(duplicate_groups), '| Extra records:', int((duplicate_groups-1).sum()))
print('Exact repeated selected-field records:', int(history[list(POSITIONS)].duplicated().sum()))
print('Delinquency codes (row counts):')
print(history.delinq.replace('', '<blank>').value_counts().to_string())
print('Zero balance codes (row counts):')
print(history.zero_code.replace('', '<blank>').value_counts().to_string())
assert len(duplicate_groups)==0
assert history.report_idx.notna().all()
known_codes = ADVERSE | NON_ADVERSE | REPURCHASE
unexpected = sorted(set(history.zero_code) - known_codes - {''})
assert not unexpected, f'Review unexpected terminal codes: {unexpected}'


report blank: 0 invalid nonblank: 0
orig blank: 0 invalid nonblank: 0
first_payment blank: 0 invalid nonblank: 0
zero_month blank: 242005 invalid nonblank: 0
Duplicate loan-month groups: 0 | Extra records: 0
Exact repeated selected-field records: 0
Delinquency codes (row counts):
delinq
00    237582
01      2676
XX       974
02       939
03       528
04       411
05       354
06       304
07       222
08       200
09       187
10       152
11       130
12       128
13       106
14        95
15        73
16        64
17        58
18        50
19        47
20        37
21        29
23        28
22        19
24        18
25        13
29        12
26        11
27         9
28         9
30         9
31         8
32         8
33         8
35         8
34         7
36         6
37         5
38         5
39         4
40         4
41         3
42         3
43         2
44         1
45         1
46         1
47         1
48         1
49         1
50         1
Zero balance codes (row counts):
zer

## 5. Compare loan-level timing and eligibility
Do not choose a date when records disagree. Compare reporting and effective terminal dates, distinguish unknowns at termination from active-month unknowns, and count how stricter observation rules would change eligibility. No final target column is created.

In [5]:
audit = []
for loan_id, g in history.groupby('loan_id', sort=False):
    g = g.sort_values('report_idx')
    orig = g.orig_idx.dropna().unique()
    fp = g.first_payment_idx.dropna().unique()
    row = dict(orig_conflict=len(orig)!=1, fp_conflict=len(fp)!=1)
    if len(orig)!=1 or len(fp)!=1 or g.orig_idx.isna().any() or g.first_payment_idx.isna().any():
        row['date_resolved'] = False
        audit.append(row)
        continue
    row['date_resolved'] = True
    start = int(fp[0]); end = start+23
    row['origination_quarter'] = str(pd.Period(year=int(orig[0])//12, month=int(orig[0])%12+1, freq='M').asfreq('Q'))
    w = g[g.report_idx.between(start,end)]
    terminal = g[g.zero_code.ne('')]
    tr = terminal[terminal.report_idx.between(start,end)]
    te = terminal[terminal.zero_month_idx.between(start,end).fillna(False)]
    row['left_truncated'] = int(g.report_idx.min()) > start
    row['followup_complete'] = int(g.report_idx.max()) >= end or not tr.empty
    row['candidate_freddie_eligible'] = not row['left_truncated'] and row['followup_complete']
    row['terminal_report_in_window'] = not tr.empty
    row['terminal_effective_in_window'] = not te.empty
    row['terminal_window_disagrees'] = (not tr.empty) != (not te.empty)
    row['terminal_date_differs'] = bool((terminal.report_idx != terminal.zero_month_idx).fillna(True).any())
    row['candidate_adverse_report'] = bool(tr.zero_code.isin(ADVERSE).any())
    row['candidate_adverse_effective'] = bool(te.zero_code.isin(ADVERSE).any())
    row['candidate_serious_numeric'] = bool(w.numeric_delinq.ge(3).any())
    row['repurchase_in_window'] = bool(tr.zero_code.isin(REPURCHASE).any())
    unknown = w.numeric_delinq.isna()
    row['unknown_any'] = bool(unknown.any())
    row['unknown_terminal_row'] = bool((unknown & w.zero_code.ne('')).any())
    row['unknown_active_row'] = bool((unknown & w.zero_code.eq('')).any())
    last_required = min(end, int(tr.report_idx.min())) if not tr.empty else end
    required = set(range(start,last_required+1))
    observed = set(w.report_idx.astype(int))
    row['missing_required_months'] = len(required-observed)
    row['candidate_strict_eligible'] = row['candidate_freddie_eligible'] and not row['missing_required_months'] and not row['unknown_active_row'] and not row['repurchase_in_window']
    row['terminal_code_in_window'] = '|'.join(sorted(set(tr.zero_code))) or '<none>'
    audit.append(row)
audit = pd.DataFrame(audit)
assert len(audit)==4749
resolved = audit[audit.date_resolved].copy()
print('Resolved dates:', len(resolved), 'Unresolved:', len(audit)-len(resolved))


Resolved dates: 4744 Unresolved: 5


## 6. Summarize diagnostic comparisons
Count unresolved loans separately. Compare candidate eligibility and event flags without treating unknown statuses as negative outcomes. No loan IDs are exported.

In [6]:
checks = []
for name in ['orig_conflict','fp_conflict','date_resolved']:
    checks.append({'check':name,'count':int(audit[name].eq(True).sum()),'denominator':len(audit)})
for name in resolved.columns:
    if name not in ['orig_conflict','fp_conflict','date_resolved','origination_quarter','terminal_code_in_window','missing_required_months']:
        checks.append({'check':name,'count':int(resolved[name].eq(True).sum()),'denominator':len(resolved)})
checks = pd.DataFrame(checks)
print(checks.to_string(index=False))
comparison = resolved.groupby(['candidate_freddie_eligible','candidate_strict_eligible','candidate_serious_numeric','candidate_adverse_report','unknown_active_row','unknown_terminal_row','repurchase_in_window'],dropna=False).size().reset_index(name='loans')
terminal_counts = resolved.groupby(['terminal_code_in_window','terminal_window_disagrees','terminal_date_differs'],dropna=False).size().reset_index(name='loans')
cohort_counts = resolved.groupby(['origination_quarter','candidate_freddie_eligible','candidate_strict_eligible']).size().reset_index(name='loans')
conflict_patterns = audit[~audit.date_resolved].groupby(['orig_conflict','fp_conflict']).size().reset_index(name='loans')
coverage = resolved.groupby('missing_required_months').size().reset_index(name='loans')
assert comparison.loans.sum()==terminal_counts.loans.sum()==cohort_counts.loans.sum()==coverage.loans.sum()==len(resolved)
print(comparison.to_string(index=False))

                       check  count  denominator
               orig_conflict      4         4749
                 fp_conflict      1         4749
               date_resolved   4744         4749
              left_truncated    252         4744
           followup_complete   4743         4744
  candidate_freddie_eligible   4491         4744
   terminal_report_in_window    884         4744
terminal_effective_in_window    884         4744
   terminal_window_disagrees      0         4744
       terminal_date_differs      0         4744
    candidate_adverse_report      3         4744
 candidate_adverse_effective      3         4744
   candidate_serious_numeric     35         4744
        repurchase_in_window      3         4744
                 unknown_any    876         4744
        unknown_terminal_row    876         4744
          unknown_active_row      0         4744
   candidate_strict_eligible   4488         4744
 candidate_freddie_eligible  candidate_strict_eligible  candidate_ser

## 7. Export the review bundle
Save aggregate diagnostic results only. Review unresolved dates, unknown statuses, repurchases and event timing before implementing Stage 3D. The strict eligibility flag is an exploratory alternative, not the selected rule.

In [7]:
out = ROOT/'data/processed/Fannie_Mae/notebook_14c_target_validation'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
for name,table in {'candidate_code_map.csv':code_map,'eligibility_checks.csv':checks,'eligibility_event_comparison.csv':comparison,'terminal_timing_comparison.csv':terminal_counts,'cohort_eligibility.csv':cohort_counts,'date_conflict_patterns.csv':conflict_patterns,'required_month_gaps.csv':coverage}.items():
    table.to_csv(out/name,index=False)
meta = dict(stage='3A–3C',source_name=SOURCE.name,file_bytes=SIZE,physical_rows=int(scan_counts['physical_rows']),sample_rows=len(history),sample_loans=len(audit),sample_modulus=SAMPLE_MODULUS,freddie_source_sha256=freddie_sha,window='first payment +0 through +23',candidate_adverse_codes=sorted(ADVERSE),release_vintage='unverified',final_target_constructed=False,models_scored=False)
(out/'stage3_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
zpath=out/'notebook_14c_stage3abc_review.zip'
with zipfile.ZipFile(zpath,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        if f!=zpath: z.write(f,f.name)
print('Review ZIP:',zpath)

Review ZIP: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_14c_target_validation\20261124T145529086616Z\notebook_14c_stage3abc_review.zip


## Review gate
Upload the ZIP. Confirm the local glossary release and decide how to handle unresolved dates, unknown delinquency, repurchases and terminal timing before constructing labels. Do not push yet.